# Country Missingness Scoring

Identifies countries systematically absent from data they should have.
Classifies each (country, year) into 10 statuses: dissolved, political_exclusion,
self_exclusion, nascent, collision, microstate, failed, degraded, reporting, strong.

**Phase 1: Model Definition**

See `phase1/document/country_missingness_methodology.md` for full methodology.

In [1]:
using Revise
using InteractiveUtils

includet("phase1/functions/load_phase1.jl")

╔══════════════════════════════════════════════════════════════════════════╗
║ QoG METADATA JOINING - PHASE 0 LOADED                                ║
╚══════════════════════════════════════════════════════════════════════════╝

Quick Start:
    metadata = join_metadata()              # Run full pipeline (single isomorphism check)
    metadata = join_metadata_with_cascade()  # Run cascade (strictest → loosest), then union on slug
    quick_check()                             # Diagnostic check
    inspect_exceptions()                      # Review configuration
    show_usage()                              # Detailed documentation

Pipeline Steps:
    1. ingest_and_normalize()            # Load & normalize sources (PDF = qog_slugs_temporal.csv; min_year/max_year ingested)
    2. align_id_variables!(...)          # Harmonize ID vars
    3. run_isomorphism_cascade(...)      # Strictest → loosest until success; returns (stata_df, pdf_df, arrow_df) for union on slug
    4. unify_and_join(..

In [2]:
using CSV, DataFrames

df = load_augmented_or_build()
meta_df = CSV.read("data/qog_metadata_plus2.csv", DataFrame)
println("Loaded: $(nrow(df)) rows, $(nrow(meta_df)) slugs")

✓ Checksum verified: data/qog_std_ts_jan25_aug.arrow
✓ Loaded: 12391 rows × 2014 cols from data/qog_std_ts_jan25_aug.arrow
  ggis_rowid unique: ✓ | Required columns: ✓ | Missing regions: 0 ✓
Loaded: 12391 rows, 2010 slugs


## Run Pipeline

In [3]:
result = run_country_missingness(df, meta_df)

Step 0 — Country Temporal Profiles
    Total countries: 200
    Active: 194
    Dissolved: 6
      CSK Czechoslovakia (dissolved 1992, last data 1992)
      DDR German Democratic Republic (dissolved 1990, last data 1990)
      YMD Yemen Democratic (dissolved 1990, last data 1989)
      SUN USSR (dissolved 1991, last data 1991)
      YUG Yugoslavia (dissolved 1992, last data 1991)
      XTI Tibet (dissolved 1959, last data 1950)
    Microstates: 11
      AND Andorra (pop 84K)
      ATG Antigua and Barbuda (pop 93K)
      DMA Dominica (pop 73K)
      LIE Liechtenstein (pop 39K)
      MCO Monaco (pop 39K)
      NRU Nauru (pop 12K)
      MHL Marshall Islands (pop 52K)
      PLW Palau (pop 20K)
      KNA Saint Kitts and Nevis (pop 47K)
      SMR San Marino (pop 35K)
      TUV Tuvalu (pop 11K)
    Political exclusion: 1
      TWN Taiwan (Province of China)
    Self-exclusion: 2
      PRK Korea (the Democratic People's Republic of)
      SUN USSR

Step 1 — Per-Country, Per-Year Global Slug Co

(profiles = 200×19 DataFrame
 Row │ ident_ccode  ident_ccodealp  ident_cname                        country ⋯
     │ Int64        String          String                             Int64   ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │           4  AFG             Afghanistan                                ⋯
   2 │           8  ALB             Albania
   3 │          12  DZA             Algeria
   4 │          20  AND             Andorra
   5 │          24  AGO             Angola                                     ⋯
   6 │          28  ATG             Antigua and Barbuda
   7 │          31  AZE             Azerbaijan
   8 │          32  ARG             Argentina
   9 │          36  AUS             Australia                                  ⋯
  10 │          40  AUT             Austria
  11 │          44  BHS             Bahamas (the)
  ⋮  │      ⋮             ⋮                         ⋮                          ⋱
 191 │         840  USA      

## Status Distribution

In [4]:
sort(combine(groupby(result.status, :country_status), nrow => :count), :count, rev=true)

Row,country_status,count
,String,Int64
1,strong,7408
2,reporting,3189
3,nascent,990
4,microstate,562
5,self_exclusion,78
6,political_exclusion,52
7,failed,42
8,dissolved,28
9,collision,17


## Dissolved States

In [5]:
filter(r -> r.is_dissolved, result.profiles)[:, [:ident_ccodealp, :ident_cname, :country_birth_year, :country_death_year, :dissolution_year]]

Row,ident_ccodealp,ident_cname,country_birth_year,country_death_year,dissolution_year
,String,String,Int64,Int64,Int64?
1,CSK,Czechoslovakia,1946,1992,1992
2,DDR,German Democratic Republic,1949,1990,1990
3,YMD,Yemen Democratic,1968,1989,1990
4,SUN,USSR,1946,1991,1991
5,YUG,Yugoslavia,1946,1991,1992
6,XTI,Tibet,1946,1950,1959


## Political Exclusion

Entities excluded BY others from international data programs (not state failure).

In [6]:
pol_excl = filter(r -> r.is_political_exclusion, result.profiles)
if nrow(pol_excl) > 0
    for r in eachrow(pol_excl)
        println("  $(r.ident_ccodealp) $(r.ident_cname) (from $(r.pol_excl_from))")
        # Show coverage trajectory
        rows = filter(row -> row.ident_ccode == r.ident_ccode, result.status)
        for decade in [1990, 2000, 2010, 2020]
            dec = filter(row -> decade <= row.ident_year < decade + 10, rows)
            nrow(dec) == 0 && continue
            avg = round(mean(dec.global_coverage_pct) * 100, digits=1)
            sts = join(unique(dec.country_status), "/")
            println("    $(decade)s: $sts ($avg% avg)")
        end
    end
else
    println("  None")
end

  TWN Taiwan (Province of China) (from 1950)
    1990s: reporting/political_exclusion (42.3% avg)
    2000s: reporting (46.9% avg)
    2010s: reporting/political_exclusion (41.0% avg)
    2020s: political_exclusion (20.0% avg)


## Self-Exclusion

Entities excluding THEMSELVES from international data engagement.

In [7]:
self_excl = filter(r -> r.is_self_exclusion, result.profiles)
if nrow(self_excl) > 0
    for r in eachrow(self_excl)
        println("  $(r.ident_ccodealp) $(r.ident_cname) (from $(r.self_excl_from))")
        rows = filter(row -> row.ident_ccode == r.ident_ccode, result.status)
        for decade in [1960, 1970, 1980, 1990, 2000, 2010, 2020]
            dec = filter(row -> decade <= row.ident_year < decade + 10, rows)
            nrow(dec) == 0 && continue
            avg = round(mean(dec.global_coverage_pct) * 100, digits=1)
            sts = join(unique(dec.country_status), "/")
            println("    $(decade)s: $sts ($avg% avg)")
        end
    end
else
    println("  None")
end

  PRK Korea (the Democratic People's Republic of) (from 1950)
    1960s: self_exclusion (29.3% avg)
    1970s: self_exclusion (35.5% avg)
    1980s: self_exclusion/reporting (39.0% avg)
    1990s: reporting (53.2% avg)
    2000s: reporting (61.5% avg)
    2010s: reporting (55.1% avg)
    2020s: reporting/self_exclusion (27.4% avg)
  SUN USSR (from 1950)
    1960s: self_exclusion (5.8% avg)
    1970s: self_exclusion (8.1% avg)
    1980s: self_exclusion/dissolved (10.7% avg)
    1990s: dissolved (9.7% avg)


## Microstates (pop < 100K)

In [8]:
micros = filter(r -> r.is_microstate, result.profiles)
println("$(nrow(micros)) microstates:")
for r in eachrow(micros)
    pop_str = ismissing(r.max_pop) ? "?" : "$(Int(round(r.max_pop)))K"
    println("  $(r.ident_ccodealp) $(rpad(r.ident_cname, 30)) pop $pop_str")
end

11 microstates:
  AND Andorra                        pop 84K
  ATG Antigua and Barbuda            pop 93K
  DMA Dominica                       pop 73K
  LIE Liechtenstein                  pop 39K
  MCO Monaco                         pop 39K
  NRU Nauru                          pop 12K
  MHL Marshall Islands               pop 52K
  PLW Palau                          pop 20K
  KNA Saint Kitts and Nevis          pop 47K
  SMR San Marino                     pop 35K
  TUV Tuvalu                         pop 11K


## Failed Countries

Active countries with <40% global slug coverage AND >20ppt below subregion peers.
Should NOT include dissolved, nascent, collision, or exclusion entities.

In [10]:
failed = filter(r -> r.country_status == "failed", result.status)
failed_countries = unique(failed.ident_ccode)
println("Countries with 'failed' years: $(length(failed_countries))\n")
for ccode in failed_countries
    prof = filter(r -> r.ident_ccode == ccode, result.profiles)
    rows = filter(r -> r.ident_ccode == ccode, result.status)
    alpha = prof.ident_ccodealp[1]
    name = prof.ident_cname[1]
    fyears = sort(filter(r -> r.country_status == "failed", rows).ident_year)
    println("  $alpha $(rpad(name, 35)) $(length(fyears)) failed years: $(first(fyears))–$(last(fyears))")
    for decade in [1960, 1970, 1980, 1990, 2000, 2010, 2020]
        dec = filter(r -> decade <= r.ident_year < decade + 10, rows)
        nrow(dec) == 0 && continue
        avg = round(mean(dec.global_coverage_pct) * 100, digits=1)
        sts = join(unique(dec.country_status), "/")
        println("    $(decade)s: $sts ($avg% avg)")
    end
    println()
end

Countries with 'failed' years: 5

  MYS Malaysia                            3 failed years: 1960–1962
    1960s: failed/strong (24.9% avg)
    1970s: strong (44.4% avg)
    1980s: strong (51.1% avg)
    1990s: strong (68.9% avg)
    2000s: strong (80.2% avg)
    2010s: strong (76.5% avg)
    2020s: strong (42.5% avg)

  SRB Serbia                              9 failed years: 1997–2005
    1990s: nascent/failed (12.9% avg)
    2000s: failed/reporting (34.8% avg)
    2010s: reporting/strong (65.6% avg)
    2020s: strong/reporting (39.1% avg)

  ZWE Zimbabwe                            6 failed years: 1960–1965
    1960s: failed/reporting (11.9% avg)
    1970s: reporting (36.8% avg)
    1980s: reporting/strong (51.5% avg)
    1990s: strong (68.3% avg)
    2000s: strong (79.7% avg)
    2010s: strong (76.6% avg)
    2020s: strong (42.5% avg)

  YMD Yemen Democratic                    12 failed years: 1973–1984
    1960s: nascent (16.8% avg)
    1970s: nascent/failed (19.5% avg)
    1980s: fa

## Degraded Countries

Coverage dropping relative to self AND peers (not just global reporting lag).

In [11]:
degraded = filter(r -> r.country_status == "degraded", result.status)
degraded_countries = unique(degraded.ident_ccode)
println("Countries with 'degraded' years: $(length(degraded_countries))\n")
for ccode in degraded_countries
    prof = filter(r -> r.ident_ccode == ccode, result.profiles)
    rows = filter(r -> r.ident_ccode == ccode, result.status)
    alpha = prof.ident_ccodealp[1]
    name = prof.ident_cname[1]
    dyears = sort(filter(r -> r.country_status == "degraded", rows).ident_year)
    println("  $alpha $(rpad(name, 35)) $(length(dyears)) degraded years: $(first(dyears))–$(last(dyears))")
end

Countries with 'degraded' years: 3

  DZA Algeria                             1 degraded years: 2024–2024
  MAR Morocco                             1 degraded years: 2024–2024
  EGY Egypt                               1 degraded years: 2024–2024


## Nascent Countries

First 5 years of data — newly independent, building data infrastructure.

In [12]:
nascent = filter(r -> r.country_status == "nascent", result.status)
nascent_countries = unique(nascent.ident_ccode)
println("Countries with 'nascent' years: $(length(nascent_countries))")

Countries with 'nascent' years: 199


## Revised Slug Penetration

Population-weighted penetration recalculated with clean denominator
(excluding dissolved, micro, failed, exclusion, collision country-years).

In [13]:
original_global = count(r -> r.original_penetration >= 0.95, eachrow(result.penetration))
revised_global = count(r -> r.revised_penetration >= 0.95, eachrow(result.penetration))
println("Slugs ≥95% penetration:")
println("  Original denominator: $original_global")
println("  Revised denominator:  $revised_global")
println("  New globals:          $(revised_global - original_global)")

Slugs ≥95% penetration:
  Original denominator: 336
  Revised denominator:  581
  New globals:          245


In [ ]:
# Top gainers
first(result.penetration, 20)

## Save Flags

In [ ]:
# CSV.write("data/country_missingness_flags.csv", result.flags)
# println("\u2705 Saved country_missingness_flags.csv")